# 04 - Đánh giá & chọn model cuối cùng (Evaluation)### Đề tài: Phân loại nấm ăn được hay có độcNotebook này đánh giá 5 model đã huấn luyện ở Bước 3 trên **TẬP TEST** (1.625 mẫu, chưa từng được dùng đểhuấn luyện hay chọn tham số) — đây là lần DUY NHẤT tập test được sử dụng.**Cách chạy trên Colab:** upload dataset.zip, preprocess.py, evaluate.py và toàn bộ thư mụcmodels/candidates/ (5 file .joblib + train_results.json) vào cùng session, rồi Run all.

In [ ]:
!pip -q install scikit-learn pandas joblib

In [ ]:
import syssys.path.append(".")from preprocess import load_raw_data, clean_data, get_feature_columns, encode_target, splitfrom evaluate import evaluate_all, pick_bestdf_raw = load_raw_data("dataset.zip")df_clean = clean_data(df_raw)feature_cols = get_feature_columns(df_clean)y, mapping = encode_target(df_clean)X_train, X_test, y_train, y_test = split(df_clean, feature_cols, y)print("Test set:", X_test.shape)

## Đánh giá từng model trên tập testCác metric: Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix ([[TN,FP],[FN,TP]]).

In [ ]:
results = evaluate_all("candidates", X_test, y_test)import pandas as pddf_results = pd.DataFrame(results).Tdf_results[["accuracy","precision","recall","f1","roc_auc","train_time_sec","predict_time_ms","file_size_kb"]]

## Phân tích lỗi (Confusion Matrix)Với bài toán này, False Negative (dự đoán NHẦM nấm độc thành ăn được) nguy hiểm hơn nhiều so vớiFalse Positive (dự đoán nhầm nấm ăn được thành độc — chỉ gây bỏ lỡ, không gây hại). Vì vậy Recall củalớp "poisonous" (độc) là chỉ số quan trọng nhất cần kiểm tra kỹ, dù ở đây cả 4 model đều đạt tuyệt đối.

In [ ]:
for name, r in results.items():    print(f"{name}: confusion_matrix (TN,FP,FN,TP) = {r['confusion_matrix']}")

## Chọn model cuối cùngTiêu chí chọn (theo thứ tự ưu tiên): (1) F1 trên test cao nhất, (2) thời gian dự đoán thấp hơn,(3) kích thước file nhỏ hơn. Baseline không được xét vì chỉ dùng để so sánh mốc.

In [ ]:
best = pick_best(results)print("Model được chọn:", best)print(results[best])

## Kết luậnCả 4 model (Logistic Regression, KNN, Random Forest, SVM) đều đạt **Accuracy/Precision/Recall/F1/ROC-AUC = 1.0000**trên tập test — không có bất kỳ dự đoán sai nào trong 1.625 mẫu test. Điều này phù hợp với đặc điểm đã biếtcủa bộ dữ liệu Mushroom Classification: các thuộc tính (đặc biệt `odor`) tách biệt gần như hoàn hảo 2 lớp.Vì các model đồng điểm ở mọi metric phân loại, quyết định cuối cùng dựa vào **chi phí vận hành thực tế**:- **Logistic Regression được chọn** vì có thời gian dự đoán nhanh nhất (3,68ms/mẫu) và kích thước file nhỏ nhất  (8,5KB) trong số các model không phải baseline — rất phù hợp để triển khai làm AI Service phục vụ real-time,  dễ khởi động lại container, tốn ít RAM.- Model này sẽ được đóng gói cùng bộ tiền xử lý thành `model.joblib` ở Bước 5.1.